<br>

# Phase 3 — grounding & the converging-evidence decision

---

This notebook executes steps 3–5 of the annotation loop from `docs/design.md` — the heart of zlabel:

1. Normalize symbols *(Phase 2)*
2. Score against panels *(Phase 2)*
3. **Ground** the top markers in vivo (ZFIN expression → ZFA anatomy; plausible for the stage?)
4. **Decide** — assign a bucket with confidence, roll up to a coarser tier, or abstain honestly
5. **Emit** a `Label` evidence packet

Phase 2 produced a ranked bucket table. Phase 3 turns that table into a decision: one
`Labeler(stage_hpf=…).label([...])` call runs the whole loop and returns a `Label` — or an honest
`mixed/unresolved` when the evidence does not converge.

Need the scoring walkthrough first? See [Phase 2 notebook](phase_02.ipynb).

<div class="alert alert-warning" role="alert">
    <b>⚙️ Prerequisite — run before this notebook</b>
    <br><br>
    <b>1.</b> From the repo root: <code>bash scripts/setup_data.sh</code> &nbsp;(downloads ontology files into <code>data/ontologies/</code>)
    <br>
    <b>2.</b> Start the server with <code>make notebook</code> &nbsp;(keeps the working directory at the repo root)
</div>

In [1]:
import os
from pathlib import Path

import zlabel
from zlabel.ground import expression_lookup, grounds_under

print(f"zlabel {zlabel.__version__}")

# Resolve data/ontologies relative to the repo root (the server starts there via `make notebook`).
PACKAGE = "zlabel"
ROOT_DIR = Path(os.getcwd().split(PACKAGE, 1)[0]) / PACKAGE
DATA_DIR = ROOT_DIR / "data/ontologies"
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data not found at {DATA_DIR.resolve()}.  Run: bash scripts/setup_data.sh"
    )

# Labeler bundles the three data authorities + panels.yaml behind one constructor. We also
# load ZFA + ZFIN expression standalone so section 1 can show the grounding primitives directly.
zfa = zlabel.load_zfa(DATA_DIR / "zfa.obo")
expr = zlabel.load_zfin_expression(DATA_DIR / "zfin_wildtype_expression.txt")
panels = zlabel.load_panels(Path(zlabel.__file__).parent / "panels.yaml")
muscle_anchor = next(p for p in panels if p.bucket == "muscle").ontology_anchor

lab = zlabel.Labeler(stage_hpf=48)  # 48 hpf = Long-pec, the keystone sample stage
print(f"ZFA terms: {zfa.number_of_nodes():,}  ·  genes with expression: {len(expr):,}")
print(f"Labeler ready (stage_hpf=48).  muscle anchor = {set(muscle_anchor)}")

zlabel 0.1.0


ZFA terms: 3,161  ·  genes with expression: 14,485
Labeler ready (stage_hpf=48).  muscle anchor = {'ZFA:0000548'}


## 1. Grounding — does a marker express where the bucket says it should?

A panel score says *which* bucket a cluster's markers belong to. **Grounding** checks that claim
against reality: do those markers actually express in the right anatomy in a live zebrafish?

Two pure functions do the work:

- **`expression_lookup(expr, symbol)`** — fetch a gene's curated ZFIN wildtype-expression records
  (each one a ZFA anatomy term + a developmental-stage range).
- **`grounds_under(zfa, zfa_id, anchor)`** — walk the ZFA `is_a` / `part_of` graph to test whether
  an expression site sits at or under the bucket's anchor term.

Grounding is **anchor-relative**: the same marker grounds under its own tissue and not under others.

In [2]:
# myod1 is a muscle master regulator. Its in-vivo expression records land in muscle anatomy.
recs = expression_lookup(expr, "myod1")
print(f"myod1 has {len(recs):,} ZFIN expression records.  A few of the anatomy terms:")
seen = []
for r in recs:
    if r.zfa_id not in seen:
        seen.append(r.zfa_id)
        print(f"   {r.zfa_id}  {zlabel.term_name(zfa, r.zfa_id)}")
    if len(seen) == 4:
        break

# Anchor-relative: myod1 grounds under muscle, not under blood.
blood_anchor = next(p for p in panels if p.bucket == "blood_erythroid").ontology_anchor
under_muscle = any(grounds_under(zfa, r.zfa_id, muscle_anchor) for r in recs)
under_blood = any(grounds_under(zfa, r.zfa_id, blood_anchor) for r in recs)
print(f"\nmyod1 grounds under muscle anchor {set(muscle_anchor)}: {under_muscle}")
print(f"myod1 grounds under blood anchor  {set(blood_anchor)}: {under_blood}")

myod1 has 960 ZFIN expression records.  A few of the anatomy terms:
   ZFA:0000003  adaxial cell
   ZFA:0001058  caudal fin
   ZFA:0009117  fast muscle cell
   ZFA:0001114  head

myod1 grounds under muscle anchor {'ZFA:0000548'}: True
myod1 grounds under blood anchor  {'ZFA:0000007', 'ZFA:0005023'}: False


## 2. The decision — the whole loop in one call

`Labeler.label()` normalizes the markers, scores them, grounds the winner, checks the stage, and
emits a `Label` evidence packet. Here is the keystone 48 hpf muscle cluster (note the input uses the
old symbol `mylz2`, which normalizes to `mylpfa` before scoring).

In [ ]:
KEYSTONE = ["mylz2", "acta1b", "tnnt3a", "myod1", "myog"]  # a 48 hpf fast-muscle cluster
label = lab.label(KEYSTONE)

levels_str = " -> ".join(label.levels) if label.levels else "()"
print(f"bucket           : {label.bucket}")           # named ZFA term, not the panel bucket
print(f"depth            : {label.depth}")            # len(levels); evidence-derived
print(f"levels           : {levels_str}")
print(f"panel_bucket     : {label.panel_bucket}")     # coarse prior that anchored the vote
print(f"panel_germ_layer : {label.panel_germ_layer}")
print(f"convergent_genes : {label.convergent_genes}")  # markers that voted for the named ZFA term
print(f"confidence       : {label.confidence}  (score {label.confidence_score:.3f})")
print(f"zfa_id           : {label.zfa_id}")
print(f"next_step        : {label.next_step}")
print(f"rationale        : {label.rationale}")
print("
expression evidence — each supporting marker's grounded anatomy:")
for hit in label.expression_evidence:
    print(f"   {hit.symbol:<8} -> {hit.zfa_id}  {hit.zfa_name}")


## 3. The confidence rubric

Confidence is a weighted 0–1 score over four components, each in `[0, 1]`:

- **coherence** (0.40) — rank-weighted strength of the winner’s matched markers
- **margin** (0.30) — how far the winner leads the runner-up bucket
- **grounding** (0.20) — fraction of the winning panel’s matched markers whose ZFIN
  expression grounds under the named ZFA term (or the panel anchor when the vote fails)
- **stage** (0.10) — fraction of those markers on-stage for the sample

For the keystone cluster every component is 1.0 — strong panel, dominant lead, all convergent
markers ground to muscle cell anatomy, all on-stage at Long-pec — so it earns a clean `high`.

In [4]:
WEIGHTS = {"coherence": 0.40, "margin": 0.30, "grounding": 0.20, "stage": 0.10}

print(f"{'component':<11} {'weight':>6}  {'value':>5}  contribution")
print("-" * 48)
for name, w in WEIGHTS.items():
    v = label.confidence_components[name]
    bar = "#" * round(v * 20)
    print(f"{name:<11} {w:>6.2f}  {v:>5.2f}  {bar} {w * v:.2f}")
print("-" * 48)
print(f"{'score':<11} {'':>6}  {'':>5}  -> {label.confidence_score:.2f}  ({label.confidence})")

print("\nTiers:  >= 0.80 high  ·  >= 0.60 medium  ·  else low")
print("Caps (honest by design): a germ-layer rollup never exceeds medium, and high needs real")
print("grounding/stage corroboration — anatomy that contradicts the call blocks high regardless of")
print("stage. The weights are provisional; Phase 4 eval calibrates them.")

component   weight  value  contribution
------------------------------------------------
coherence     0.40   1.00  #################### 0.40
margin        0.30   1.00  #################### 0.30
grounding     0.20   1.00  #################### 0.20
stage         0.10   1.00  #################### 0.10
------------------------------------------------
score                      -> 1.00  (high)

Tiers:  >= 0.80 high  ·  >= 0.60 medium  ·  else low
Caps (honest by design): a germ-layer rollup never exceeds medium, and high needs real
grounding/stage corroboration — anatomy that contradicts the call blocks high regardless of
stage. The weights are provisional; Phase 4 eval calibrates them.


## 4. It never overcalls

The reason to ground at all is to know when *not* to commit. The same `label()` call abstains or
rolls up when the evidence does not converge on one bucket:

- **abstain** (`mixed/unresolved`) — the top contenders span contradictory germ layers.
- **roll up** (`underclustered`) — no single bucket dominates, but the contenders share a germ layer,
  so it backs off to that coarser, honest tier instead of guessing.
- **assign anyway, at lower confidence** — a single weak-but-real marker is still labeled, just not `high`.

In [5]:
cases = {
    "confident (muscle)":      ["mylz2", "acta1b", "tnnt3a", "myod1", "myog"],
    "abstain (mixed)":         ["elavl3", "neurod1", "gata1a", "hbae1.1"],
    "rollup (underclustered)": ["myod1", "myog", "gata1a", "hbae1.1"],
    "weak single signal":      ["elavl3"],
}

print(f"{'case':<26} {'bucket':<17} {'confidence':<11} {'flag':<15} markers")
print("-" * 98)
for name, markers in cases.items():
    r = lab.label(markers)
    print(f"{name:<26} {r.bucket:<17} {(r.confidence or '—'):<11} {r.ambiguity_flag:<15} {', '.join(markers)}")

print("\nneural + blood markers span contradictory germ layers -> honest mixed/unresolved.")
print("muscle + blood share the mesoderm germ layer but neither dominates -> roll up to mesoderm.")
print("a lone neural marker is still assigned (neural) — at medium, not high.")

case                       bucket            confidence  flag            markers
--------------------------------------------------------------------------------------------------
confident (muscle)         muscle            high        none            mylz2, acta1b, tnnt3a, myod1, myog
abstain (mixed)            mixed/unresolved  —           mixed           elavl3, neurod1, gata1a, hbae1.1
rollup (underclustered)    mesoderm          medium      underclustered  myod1, myog, gata1a, hbae1.1
weak single signal         neural            medium      none            elavl3

neural + blood markers span contradictory germ layers -> honest mixed/unresolved.
muscle + blood share the mesoderm germ layer but neither dominates -> roll up to mesoderm.
a lone neural marker is still assigned (neural) — at medium, not high.


## 5. Identity vs. state (optional)

Cell **identity** (what a cell is) and cell **state** (what it is doing) are orthogonal. A dividing
muscle cell is still muscle, so detected state programs are reported *alongside* the identity call,
never in competition with it.

In [6]:
# Muscle markers + cell-cycle markers (mki67, pcna, top2a hit the cycling state panel).
r = lab.label(["mylz2", "acta1b", "myod1", "mki67", "pcna", "top2a"])
print(f"bucket : {r.bucket}  ({r.confidence})")
print(f"states : {r.states}")
print("\nThe cycling program is reported as a state; the identity call stays muscle.")

bucket : muscle  (high)
states : ('cycling',)

The cycling program is reported as a state; the identity call stays muscle.


## Phase 3 synthesis — the handoff artifact

The deterministic annotation loop now runs end to end. The handoff artifact is the **`Label` evidence
packet**: a bucket (or honest abstention), a confidence tier with its four-component breakdown, the
grounded in-vivo evidence, and a one-line rationale — everything a human or a future LLM narrator
needs to trust or question the call, with no library internals required.

## What’s next

**Phase 4b — evaluation.** Phase 4a (this notebook’s engine) added the IC-weighted ZFA convergence
namer; panels are now the coarse prior, not the naming authority. Phase 4b runs `label()` across
a labeled atlas (Daniocell’s broad tissues) and reports the numbers that prove it works:
broad-bucket agreement, resolution depth, coverage (non-abstain rate), and abstention calibration.
Phase 4b is also where the provisional confidence weights get calibrated against ground truth.

Need a refresher on the scored bucket table that feeds this notebook? Revisit
[Phase 2 notebook](phase_02.ipynb).

<div class="alert alert-success" role="alert">
    <b>✅ What you have after Phase 4a</b>
    <br><br>
    The full layer-1 loop: marker genes in, a grounded <code>Label</code> evidence packet out —
    labeled at the deepest ZFA anatomy term the evidence supports, with the coarse panel prior
    kept visible, rolled up or honestly abstained when evidence does not converge.
</div>


In [7]:
# End of notebook.